# LC 543 — Diameter of Binary Tree
**Day 47 | Pattern: DFS Post-Order | Difficulty: Easy**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> The diameter through any node equals
left_depth + right_depth. DFS returns depth upward while
updating a global max at each node.
</div>

## Official Problem Statement

Given the `root` of a binary tree, return the length of the
**diameter** of the tree.

The diameter is the length of the longest path between any two
nodes. The path may or may not pass through the root.
The length of a path is the number of **edges** between nodes.

**Constraints:**
- Number of nodes: `[1, 10^4]`
- `-100 <= Node.val <= 100`

## What This Is Actually Asking

Find the longest path in the tree measured in edges.
The path can go through any node as the "turning point".
At the turning node, the path comes up from the left subtree
and up from the right subtree.
So the diameter through any node = left_depth + right_depth,
and we want the maximum of this over all nodes.

## Walk Through an Example by Hand

```
Tree:    1
        / \
       2   3
      / \
     4   5
```
- dfs(4): left=0, right=0. diameter=max(0,0+0)=0. return 1
- dfs(5): left=0, right=0. diameter=0. return 1
- dfs(2): left=1, right=1. diameter=max(0,1+1)=2. return 2
- dfs(3): left=0, right=0. diameter=2. return 1
- dfs(1): left=2, right=1. diameter=max(2,2+1)=3. return 3
- Answer = **3** (path 4→2→1→3 or 5→2→1→3)

## The Picture

```
        1          <-- dfs returns depth=3
       / \
      2   3        <-- left_depth=2, right_depth=1
     / \           diameter here = 2+1 = 3  <-- MAX!
    4   5

DFS return values (depth = edges to deepest leaf):
    4 -> 1
    5 -> 1
    2 -> 2  (1 + max(1,1))
    3 -> 1
    1 -> 3  (but diameter=3 already captured at node 2)

At each node: diameter = max(diameter, left + right)
Return:       1 + max(left, right)
```

## When To Use This Pattern

- When a tree path metric must be **computed at each node** and
  the maximum tracked globally, think DFS post-order + nonlocal.
- When a path **spans two subtrees** at a turning node,
  think left_depth + right_depth.
- When the function must return one thing (depth) but track
  another (diameter), think **nonlocal variable**.
- When the answer does NOT have to pass through root,
  think per-node evaluation.

## The Approach

Use a post-order DFS that returns the depth of each subtree.
At every node, compute the candidate diameter as
left_depth + right_depth and update a nonlocal maximum.
Return 1 + max(left_depth, right_depth) upward so the parent
can use this node's depth in its own calculation.

In [ ]:
from collections import deque
from typing import Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    cases = [
        ([1,2,3,4,5],             3),
        ([1,2],                   1),
        ([1],                     0),
        ([1,2,3,None,None,4,5],   3),
        ([1,None,2,None,3],       2),
    ]
    passed = 0
    for i, (vals, expected) in enumerate(cases):
        root = make_tree(vals)
        result = func(root)
        status = 'PASSED' if result == expected else 'FAILED'
        if status == 'PASSED':
            passed += 1
        print(f'Case {i+1}: {status} | '
              f'Expected {expected} | Got {result}')
    print(f'\n{passed}/{len(cases)} passed')

In [ ]:
def diameterOfBinaryTree(root: Optional[TreeNode]) -> int:
    """
    Return the diameter (longest edge path) of a binary tree.

    Strategy: Post-order DFS. At each node compute
    left_depth + right_depth as candidate diameter.
    Return 1 + max(left, right) as this node's depth.

    Args:
        root: Root of binary tree.
    Returns:
        Integer diameter in number of edges.
    """
    diameter = 0

    def dfs(node):
        nonlocal diameter
        # print(f'  dfs({node.val if node else None})')
        if not node:
            return 0
        # left = dfs(node.left)
        # right = dfs(node.right)
        # diameter = max(diameter, left + right)
        # print(f'  node={node.val} l={left} r={right} '
        #       f'd={diameter}')
        # return 1 + max(left, right)
        pass

    dfs(root)
    # print(f'Final diameter: {diameter}')
    return diameter

In [ ]:
# Uncomment and run when solution is ready
# test_harness(diameterOfBinaryTree)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Naive (height per node) | O(n²) | O(n) | Recomputes heights |
| DFS post-order + nonlocal | O(n) | O(n) | Single pass, optimal |
| Iterative post-order | O(n) | O(n) | Same, no recursion limit |

## Real World Connection

In **network routing**, the diameter of a network graph determines
the worst-case number of hops between any two nodes — critical
for SLA guarantees at **AWS**.
At **Citi**, the longest chain of dependent transactions in a
settlement graph determines the minimum settlement window.
In **data lineage**, the longest path through a pipeline DAG
tells engineers the critical path that determines total
pipeline runtime — the same diameter calculation.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra